<a href="https://colab.research.google.com/github/AnaraHayat/flyrank_assignment1/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


For content teams/account managers deciding which declining pages to
prioritize for a refresh, we will build a ranked list of pages from
page-level signals like days since last update, impressions, avg.
position, and CTR, scoring likelihood of continued decline measured
by Precision@K. A wrong call costs wasted refresh effort on pages
that weren't actually declining, or a missed page that keeps losing
traffic unnoticed. A plain rule isn't enough because while hand
rule (stale × visible) was strong at the top of the list, it ran out
of signal deeper down — a depth-3 tree found a different, still
readable pattern that beat it at Precision@50 (0.72 vs 0.68), even
after validating on unseen clients. We will claim only observed and
decision-support results.

I'm choosing this lane because I already have a working baseline, a
real (validated) signal, and a habit of checking my own claims
against leakage and overfitting — that's a solid foundation to build
on for the next 7 weeks.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**What decision does this improve?**
Which pages get prioritized for a content refresh this cycle, out of
a much larger pool of pages than any team has time to manually review.

**Who acts on it?**
A content team member or account manager with limited refresh
capacity — they take the ranked list and decide which pages to
actually rewrite, update, or re-optimize first.

**What does a wrong recommendation cost?**
- **False positive** (flagging a page as declining when it isn't):
  wasted refresh effort — hours spent updating a page that didn't
  need it, while a genuinely declining page waits.
- **False negative** (missing a page that's actually declining):
  the page keeps losing traffic/rankings silently until someone
  notices manually, by which point recovery is harder and slower.

Because refresh capacity is limited and finite, the cost isn't just
"being wrong" in the abstract — it's spending a scarce resource
(people's time) in the wrong place, which has a real opportunity
cost every week this ranking is off.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [6]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Number 1: scale of the problem
print(df.shape[0], "pages | declining rate:", round(df["is_declining_label"].mean(), 3))

# Number 2: my hand rule already beats random guessing
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
print("Hand rule Precision@50:", round(precision_at_k(df["hand_rule_score"], y, 50), 3))

# Number 3: a simple, readable model finds an even stronger pattern
from sklearn.tree import DecisionTreeClassifier
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]
print("Depth-3 tree Precision@50:", round(precision_at_k(tree_score, y, 50), 3))

30000 pages | declining rate: 0.542
Hand rule Precision@50: 0.68
Depth-3 tree Precision@50: 0.72


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work CAN say:**
- These pages showed an *observed* pattern of decline in the
  historical data (based on trend direction over the period covered).
- Pages matching certain characteristics (stale content, mid-range
  rank position, low CTR) were *directionally* more likely to be
  declining than pages without those characteristics.
- The ranked list is *decision-support* — it surfaces pages worth a
  human looking at, it does not make the refresh decision itself.
- The tree's split logic is *readable* — I can point to exactly which
  signals it used and why a page landed where it did in the ranking.
- Precision@K numbers describe how well the ranking performed on
  this specific dataset, including a held-out set of clients not seen
  during training.

**What this work will NEVER say:**
- That any single feature (or combination) *causes* a page to
  decline. Everything here is correlational pattern-matching, not a
  controlled experiment — I can't rule out confounding factors.
- That the model *predicts Google's algorithm* or search ranking
  behavior. It's scoring historical patterns in this dataset, not
  reverse-engineering how a search engine works.
- That a page flagged as "high priority" is *guaranteed* to be
  declining, or that refreshing it is *guaranteed* to fix anything —
  it's a probability-ranked suggestion, not a certainty.
- That results generalize beyond the kind of pages/clients in this
  dataset — precision numbers reflect this data, not universal truth.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.